In [14]:
import pandas as pd
import json
import re

# Load dataset
df = pd.read_csv("trade.csv")

# Function to clean and fix JSON formatting issues
def fix_json_format(trade_history):
    if not isinstance(trade_history, str) or trade_history.strip() == "":
        return None  # Handle empty values safely
    try:
        # Replace single quotes with double quotes
        fixed_str = trade_history.replace("'", '"')

        # Fix Python-style True/False/None to JSON-compatible true/false/null
        fixed_str = re.sub(r'\bTrue\b', 'true', fixed_str)
        fixed_str = re.sub(r'\bFalse\b', 'false', fixed_str)
        fixed_str = fixed_str.replace("None", "null")

        # Try parsing the cleaned string as JSON
        return json.loads(fixed_str)
    except json.JSONDecodeError:
        return None  # Return None if parsing fails

# Apply cleaning function to JSON column
df["Parsed_Trade_History"] = df["Trade_History"].apply(fix_json_format)

# Drop rows where parsing failed
df = df.dropna(subset=["Parsed_Trade_History"])

# Expand trade history into separate rows
df_exploded = df.explode("Parsed_Trade_History")

# Convert the dictionary values into separate columns
trade_details = df_exploded["Parsed_Trade_History"].apply(pd.Series)

# Merge back with the original dataframe (drop old columns)
df_final = pd.concat([df_exploded.drop(columns=["Trade_History", "Parsed_Trade_History"]), trade_details], axis=1)

### **Convert Relevant Columns to Numeric**
numeric_cols = ["price", "fee", "quantity", "realizedProfit"]
for col in numeric_cols:
    df_final[col] = pd.to_numeric(df_final[col], errors="coerce")

### **Handle Missing Values**
df_final = df_final.dropna(subset=["price", "quantity", "realizedProfit"])  # Drop rows with missing critical values

### **Remove Duplicates**
df_final = df_final.drop_duplicates()

# Reset index
df_final.reset_index(drop=True, inplace=True)

# Display the first few rows after cleaning
print("Data after cleaning:")
print(df_final.info())
print(df_final.head())

# Save cleaned dataset for further analysis
df_final.to_csv("cleaned_trade_data.csv", index=False)


Data after cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200026 entries, 0 to 200025
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Port_IDs             200026 non-null  int64  
 1   time                 200026 non-null  int64  
 2   symbol               200026 non-null  object 
 3   side                 200026 non-null  object 
 4   price                200026 non-null  float64
 5   fee                  200026 non-null  float64
 6   feeAsset             200026 non-null  object 
 7   quantity             200026 non-null  float64
 8   quantityAsset        200026 non-null  object 
 9   realizedProfit       200026 non-null  float64
 10  realizedProfitAsset  200026 non-null  object 
 11  baseAsset            200026 non-null  object 
 12  qty                  200026 non-null  float64
 13  positionSide         200026 non-null  object 
 14  activeBuy            200026 non-null  bool   
d

In [16]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("cleaned_trade_data.csv")

### **1️ Data Preparation**
# Combine side and positionSide to classify trades
df["trade_type"] = df["side"] + "_" + df["positionSide"]

# Ensure numeric data types
numeric_cols = ["quantity", "qty", "realizedProfit"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

# Group trades by account ID (Port_IDs)
grouped = df.groupby("Port_IDs")

### **2️ Compute Key Metrics**
def calculate_metrics(trades):
    total_positions = len(trades)  # Total trades made
    win_positions = sum(trades["realizedProfit"] > 0)  # Number of profitable trades
    win_rate = (win_positions / total_positions) * 100 if total_positions > 0 else 0  # Win Rate %

    # ROI Calculation
    total_investment = trades["quantity"].sum()  # Approximate total investment
    total_pnl = trades["realizedProfit"].sum()  # Total Profit & Loss
    roi = (total_pnl / total_investment) * 100 if total_investment > 0 else 0  # ROI %

    # Sharpe Ratio Calculation
    returns = trades["realizedProfit"].pct_change().dropna()
    sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(252) if not returns.empty else 0

    # Maximum Drawdown (MDD) Calculation
    cumulative_pnl = trades["realizedProfit"].cumsum()
    rolling_max = cumulative_pnl.cummax()
    drawdown = (cumulative_pnl - rolling_max) / rolling_max
    mdd = drawdown.min() * 100 if not drawdown.empty else 0  # Convert to percentage

    return pd.Series({
        "ROI (%)": roi,
        "PnL": total_pnl,
        "Sharpe Ratio": sharpe_ratio,
        "MDD (%)": mdd,
        "Win Rate (%)": win_rate,
        "Win Positions": win_positions,
        "Total Positions": total_positions
    })

# Apply function to each account
account_metrics = grouped.apply(calculate_metrics).reset_index()

### **3️ Ranking Algorithm**
# Define weights for ranking
weights = {
    "ROI (%)": 0.4,
    "PnL": 0.3,
    "Sharpe Ratio": 0.2,
    "Win Rate (%)": 0.1,
    "MDD (%)": -0.1  # Negative weight since high drawdown is bad
}

# Function to calculate performance score
def calculate_score(row):
    return (
        row["ROI (%)"] * weights["ROI (%)"] +
        row["PnL"] * weights["PnL"] +
        row["Sharpe Ratio"] * weights["Sharpe Ratio"] +
        row["Win Rate (%)"] * weights["Win Rate (%)"] +
        row["MDD (%)"] * weights["MDD (%)"]
    )

# Apply scoring function
account_metrics["Performance Score"] = account_metrics.apply(calculate_score, axis=1)

# Rank accounts based on Performance Score
account_metrics_sorted = account_metrics.sort_values(by="Performance Score", ascending=False)

### **4️⃣ Show Top 20 Ranked Accounts**
top_20_accounts = account_metrics_sorted.head(20)

# Display results
print("🔝 Top 20 Ranked Accounts:")
print(top_20_accounts)

# Save ranked results
top_20_accounts.to_csv("top_20_ranked_accounts.csv", index=False)


/usr/local/lib/python3.11/dist-packages/numpy/core/_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


🔝 Top 20 Ranked Accounts:
               Port_IDs    ROI (%)           PnL  Sharpe Ratio      MDD (%)  \
0   3672754654734989568   0.480363    565.547761           NaN   -46.298025   
1   3733192481840423936   0.232470   2653.591900           NaN   -79.117524   
2   3768170840939476993   8.779089    243.668899           NaN     0.000000   
3   3784403294629753856   0.340279   2493.618898           NaN   -17.964956   
4   3786761687746711808   0.282980    170.220200           NaN  -239.169219   
5   3788465932399412480   0.823698  13209.206074           NaN   -24.690319   
6   3818233536529843712   0.759374   5954.684401           NaN   -94.531275   
7   3819545518395756033   0.729788    976.856390           NaN     0.000000   
8   3826087012661391104  12.191385    532.656974           NaN    -5.034773   
9   3858510226868015873  -0.030456   -237.663592           NaN  -592.780700   
10  3865845304835489536   0.709080   1360.180115           NaN    -0.118212   
11  3878631538480067329   

<ipython-input-16-940dc385bc11>:50: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  account_metrics = grouped.apply(calculate_metrics).reset_index()
